In [119]:
import tech.tablesaw.api.Table;
import tech.tablesaw.api.ColumnType;
import tech.tablesaw.io.csv.CsvReadOptions;

String path ="C:\\Users\\thkle\\SSE554\\SSE554-Capstone-Project\\data\\1000_items_catalog_v2.csv";

/*Previously ran code to verify that data types do not violate min and maxes of numeric types listed
    Column |        Min |        Max
    price | 9.619999885559082 | 798.0599975585938
    review_score |        1.0 |        5.0
    review_count |       20.0 |     9989.0
    stock_quantity |        0.0 |     1998.0
*/
CsvReadOptions options = CsvReadOptions.builder(path)
    .columnTypesPartial(Map.of(
        "price", ColumnType.FLOAT,
        "review_score", ColumnType.FLOAT,
        "review_count", ColumnType.SHORT,
        "stock_quantity", ColumnType.SHORT
    )).build();
System.out.println( options );

Table table = Table.read().csv(options);
System.out.println(table.structure());


tech.tablesaw.io.csv.CsvReadOptions@a606c301
   Structure of 1000_items_catalog_v2.csv   
 Index  |   Column Name    |  Column Type  |
--------------------------------------------
     0  |       image_url  |       STRING  |
     1  |            name  |       STRING  |
     2  |       publisher  |       STRING  |
     3  |     description  |       STRING  |
     4  |        category  |       STRING  |
     5  |            tags  |       STRING  |
     6  |           price  |        FLOAT  |
     7  |    review_score  |        FLOAT  |
     8  |    review_count  |        SHORT  |
     9  |  stock_quantity  |        SHORT  |
    10  |      date_added  |   LOCAL_DATE  |


In [120]:
System.out.println("Row count: " + table.rowCount());
for( int x = 0; x < table.columnCount(); x++)
    System.out.println("Column " + x + ": " + table.column(x).name() + " - " + table.column(x).unique().size() + " unique values");

Row count: 1000
Column 0: image_url - 1000 unique values
Column 1: name - 1000 unique values
Column 2: publisher - 8 unique values
Column 3: description - 1000 unique values
Column 4: category - 6 unique values
Column 5: tags - 999 unique values
Column 6: price - 994 unique values
Column 7: review_score - 370 unique values
Column 8: review_count - 953 unique values
Column 9: stock_quantity - 781 unique values
Column 10: date_added - 883 unique values


In [121]:
/*
//Find min and max of numeric columns to determine if we can change to smaller types to save memory
import tech.tablesaw.api.ColumnType;
import java.util.ArrayList;
import java.util.Arrays;
import tech.tablesaw.api.NumericColumn;

ArrayList<ColumnType> numericTypes = new ArrayList<>(Arrays.asList(ColumnType.DOUBLE, ColumnType.FLOAT, ColumnType.INTEGER, ColumnType.LONG, ColumnType.SHORT));
System.out.printf("%10s | %10s | %10s | %10s%n", "Column", "Min", "Max", "Type");
for(int x = 0; x < table.columnCount(); x++) {
    if(numericTypes.contains(table.column(x).type())){   //Only go over number columns
        NumericColumn<?> numCol = (NumericColumn<?>) table.column(x);
        System.out.printf("%10s | %10s | %10s | %10s%n", numCol.name(), numCol.min(), numCol.max(), numCol.type());
    }
}
*/

In [122]:
/*
//Results above show that we can alter the doubles to floats and integers to shorts.
import tech.tablesaw.api.FloatColumn;
import tech.tablesaw.api.ShortColumn;
import tech.tablesaw.api.IntColumn;
import tech.tablesaw.api.DoubleColumn;

FloatColumn priceColFloat = table.doubleColumn("price").asFloatColumn();
FloatColumn reviewScoreColFloat = table.doubleColumn("review_score").asFloatColumn();
ShortColumn reviewCountColShort = table.intColumn("review_count").asShortColumn();
ShortColumn stockQuantityColShort = table.intColumn("stock_quantity").asShortColumn();

table.removeColumns("price", "review_score", "review_count", "stock_quantity");
table.addColumns(priceColFloat, reviewScoreColFloat, reviewCountColShort, stockQuantityColShort);
System.out.println(table.structure());

//Get numerics for first row and print types to verify
System.out.println("First row price type: " + table.floatColumn("price").get(0).getClass());
System.out.println("First row review score type: " + table.floatColumn("review_score").get(0).getClass());
System.out.println("First row review count type: " + table.shortColumn("review_count").get(0).getClass());
System.out.println("First row stock quantity type: " + table.shortColumn("stock_quantity").get(0).getClass());
*/

In [123]:
//Add an id column (short is enough for 1000 rows)
import tech.tablesaw.api.ShortColumn;
import java.util.Set;
import java.util.HashSet;

Set<Short> uniqueIds = new HashSet<>();
short min = 0;
short max = Short.MAX_VALUE;
for (int i = 0; i < table.rowCount(); i++) {
    while(true) {
        short id = (short) (Math.random() * (max - min + 1) + min); // Generate random ID between min and max
        if (!uniqueIds.contains(id)) {
            uniqueIds.add(id);
            break;
        }
    }
}
ShortColumn idCol = ShortColumn.create("id", uniqueIds.stream() );
table.addColumns(idCol); // Add the column

System.out.println("Added ID column successfully!");
System.out.println("Table now has " + table.columnCount() + " columns");
System.out.println("ID column sample: " + table.shortColumn("id").get(0) + ", " + 
                   table.shortColumn("id").get(1) + ", " + table.shortColumn("id").get(2));
System.out.println("\nUpdated table structure:");
System.out.println(table.structure());
System.out.println(table.first(5)); // Display the first 5 rows of the table
table.write().csv("C:\\Users\\thkle\\SSE554\\SSE554-Capstone-Project\\data\\1000_items_catalog_v2_optimized.csv");

Added ID column successfully!
Table now has 12 columns
ID column sample: 14337, 4098, 30722

Updated table structure:
   Structure of 1000_items_catalog_v2.csv   
 Index  |   Column Name    |  Column Type  |
--------------------------------------------
     0  |       image_url  |       STRING  |
     1  |            name  |       STRING  |
     2  |       publisher  |       STRING  |
     3  |     description  |       STRING  |
     4  |        category  |       STRING  |
     5  |            tags  |       STRING  |
     6  |           price  |        FLOAT  |
     7  |    review_score  |        FLOAT  |
     8  |    review_count  |        SHORT  |
     9  |  stock_quantity  |        SHORT  |
    10  |      date_added  |   LOCAL_DATE  |
    11  |              id  |        SHORT  |
                                                                                                                                                                                                               